In [ ]:
import gluestick as gs
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from ast import literal_eval

from utils.auth import OptiplyAuthenticator
from utils.models import *
from utils.payloads import *
from utils.actions import post_optiply, patch_optiply, delete_optiply, get_optiply
from utils.tools import (
    get_snapshot, snapshot_records, delete_from_snapshot,
    concat_columns, handle_invalid_dates, round_to_2, round_to_0,
    nan_to_none, validate_attribute, extract_remoteId,
    convert_to_bool, round_numeric_to_2, round_numeric_to_0
)
from utils.utils import clean_payload

In [ ]:
# Standard directory structure for hotglue
ROOT_DIR = os.environ.get("ROOT_DIR", ".")
INPUT_DIR = f"{ROOT_DIR}/sync-output"
OUTPUT_DIR = f"{ROOT_DIR}/etl-output"
SNAPSHOT_DIR = f"{ROOT_DIR}/snapshots"
CONFIG_DIR = f"{ROOT_DIR}/config"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

input_data = gs.read(INPUT_DIR)
print("Available streams:", list(input_data.keys()))

In [ ]:
# Get tenant config
config_path = f"{CONFIG_DIR}/tenant-config.json"
if not os.path.isfile(config_path):
    config_path = f"{ROOT_DIR}/tenant-config.json"

with open(config_path) as f:
    config = json.load(f)

api_creds = config.get("apiCredentials", {})
credentials = {
    "username": api_creds.get("username"),
    "password": api_creds.get("password"),
    "client_id": api_creds.get("client_id"),
    "client_secret": api_creds.get("client_secret"),
    "access_token": api_creds.get("access_token"),
    "refresh_token": api_creds.get("refresh_token"),
    "hotglue_test": api_creds.get("hotglue_test", False),
}

auth = OptiplyAuthenticator(credentials, config_path, OUTPUT_DIR)
_ = auth.access_token()
print("Auth OK")

In [ ]:
# Get config values
account_id = api_creds.get("account_id")
coupling_id = api_creds.get("couplingId")
sync_buy_orders = api_creds.get("sync_buy_orders", True)
sync_products = api_creds.get("sync_products", True)
sync_suppliers = api_creds.get("sync_suppliers", True)
sync_supplier_products = api_creds.get("sync_supplier_products", True)
sync_sell_orders = api_creds.get("sync_sell_orders", True)

# Warehouse codes to include for stock aggregation (exclude MAXRMA)
allowed_warehouses = api_creds.get("allowed_warehouses", ["MAXGAMING", "MAXGAMING2"])

api_creds_short = {
    "account_id": account_id,
    "couplingId": coupling_id,
}

print(f"Account: {account_id}, Coupling: {coupling_id}")
print(f"Sync: products={sync_products}, suppliers={sync_suppliers}, "
      f"supplierProducts={sync_supplier_products}, sellOrders={sync_sell_orders}, "
      f"buyOrders={sync_buy_orders}")
print(f"Allowed warehouses: {allowed_warehouses}")

# Summary counters
summary = {
    'products': {'deleted': 0, 'created': 0, 'updated': 0},
    'suppliers': {'deleted': 0, 'created': 0, 'updated': 0},
    'supplierProducts': {'deleted': 0, 'created': 0, 'updated': 0},
    'sellOrders': {'deleted': 0, 'created': 0, 'updated': 0},
    'buyOrders': {'deleted': 0, 'created': 0, 'updated': 0},
    'buyOrderLines': {'deleted': 0, 'created': 0, 'updated': 0},
}


In [ ]:
def check_sync_output(stream_name, alternatives=None):
    """Check if a stream exists in sync output, handling ._resource fork files"""
    names = [stream_name]
    if alternatives:
        names.extend(alternatives)
    for name in names:
        for key in input_data.keys():
            if key.startswith("._"):
                continue
            if name.lower() in key.lower():
                return input_data[key]
    return None


def clean_underscore_columns(df):
    """Remove columns starting/ending with underscore."""
    cols = [c for c in df.columns if not c.startswith('_') and not c.endswith('_')]
    return df[cols]


def parse_nested_json(val):
    """Parse a nested JSON string or literal_eval string into a list of dicts."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            return parsed if isinstance(parsed, list) else [parsed]
        except (json.JSONDecodeError, TypeError):
            try:
                parsed = literal_eval(val)
                return parsed if isinstance(parsed, list) else [parsed]
            except (ValueError, SyntaxError):
                return []
    return []


def get_extend_status_completed(status, is_open, is_received, rows):
    """Determine the completed datetime for an Extend PO.
    
    Returns a datetime string when the PO is fully received and closed,
    or None if still open.
    """
    status_str = str(status).strip() if pd.notna(status) else ""
    
    # Received and not open -> completed
    if status_str == "Received" and not is_open:
        # Use the latest statusChangeDate from rows as the completed date
        parsed_rows = parse_nested_json(rows)
        max_date = None
        for row in parsed_rows:
            if isinstance(row, dict):
                scd = row.get("statusChangeDate")
                if scd and pd.notna(scd):
                    dt = handle_invalid_dates(scd)
                    if not pd.isna(dt):
                        if max_date is None or str(dt) > str(max_date):
                            max_date = dt
        return str(max_date) if max_date else None
    
    return None


def get_deleted_at(status):
    """Return current datetime if status is Cancelled, else None."""
    status_str = str(status).strip() if pd.notna(status) else ""
    if status_str == "Cancelled":
        return datetime.datetime.utcnow().isoformat()
    return None

## Products
Extend Commerce Products -> Optiply Products

Products come from the `products` stream. The list endpoint returns one row per product-per-warehouse;
the tap deduplicates by productNumber but we still aggregate stock from the `warehouse_stock` JSON field
to sum availableBalance across allowed warehouses (default: MAXGAMING + MAXGAMING2, exclude MAXRMA).


In [ ]:
# Products: Load from tap output
products_raw = check_sync_output('products', ['Products'])
if products_raw is not None:
    products_raw = clean_underscore_columns(products_raw)
    print(f"Products loaded: {len(products_raw)}")
else:
    print("No products data found")


In [ ]:
# Products: Custom mapping
if products_raw is not None and len(products_raw) > 0 and sync_products:
    products = pd.DataFrame()
    products['id'] = products_raw['productNumber'].astype(str)
    products['remoteId'] = products_raw['productNumber'].astype(str)
    products['name'] = products_raw['productName'].astype(str).str[:255]

    # EAN code
    if 'gtinNumberList' in products_raw.columns:
        products['eanCode'] = products_raw['gtinNumberList'].apply(
            lambda x: str(x)[:50] if pd.notna(x) and str(x).strip() else None
        )
    else:
        products['eanCode'] = None

    # Article code (manufacturer product number)
    if 'manufacturerProductNumber' in products_raw.columns:
        products['articleCode'] = products_raw['manufacturerProductNumber'].apply(
            lambda x: str(x)[:100] if pd.notna(x) and str(x).strip() else None
        )
    else:
        products['articleCode'] = None

    # Price (purchase cost)
    if 'cost' in products_raw.columns:
        products['price'] = pd.to_numeric(products_raw['cost'], errors='coerce').fillna(0).clip(lower=0).round(2)
    else:
        products['price'] = 0.0

    # Status from enabled flag
    if 'enabled' in products_raw.columns:
        products['status'] = products_raw['enabled'].apply(
            lambda x: 'enabled' if str(x).lower() in ('true', '1', 'yes') else 'disabled'
        )
    else:
        products['status'] = 'enabled'

    # Stock level: aggregate availableBalance across allowed warehouses
    def compute_stock(warehouse_stock_json):
        entries = parse_nested_json(warehouse_stock_json)
        total = 0
        for entry in entries:
            if isinstance(entry, dict):
                wh = entry.get('warehouse', '')
                if wh in allowed_warehouses:
                    bal = entry.get('availableBalance', 0)
                    total += int(float(bal)) if bal is not None else 0
        return max(total, 0)

    if 'warehouse_stock' in products_raw.columns:
        products['stockLevel'] = products_raw['warehouse_stock'].apply(compute_stock)
    else:
        products['stockLevel'] = 0

    products['unlimitedStock'] = False

    # Dates
    if 'createDate' in products_raw.columns:
        products['created_at'] = products_raw['createDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
        products['updated_at'] = products['created_at']
    else:
        products['created_at'] = pd.NaT
        products['updated_at'] = pd.NaT

    # Keep product_suppliers JSON for downstream supplier/supplierProduct extraction
    if 'product_suppliers' in products_raw.columns:
        products['product_suppliers_json'] = products_raw['product_suppliers'].values
    else:
        products['product_suppliers_json'] = '[]'

    # Keep createDate string for supplier fallback date
    products['source_createDate'] = products_raw['createDate'].values if 'createDate' in products_raw.columns else None

    products = products.dropna(subset=['name'])
    products = products.drop_duplicates(subset=['remoteId'])

    # concat_attributes for change detection
    concat_fields = ['name', 'eanCode', 'articleCode', 'price', 'status', 'stockLevel']
    products['concat_attributes'] = concat_columns(products, concat_fields)

    print(f"Products mapped: {len(products)}")
else:
    products = None
    if not sync_products:
        print("Products sync disabled by config.")
    else:
        print("No products data found.")


In [ ]:
# Products: Snapshot comparison and API requests
if products is not None and len(products) > 0:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)

    # Columns for snapshot/API (exclude helper columns)
    product_api_cols = ['id', 'remoteId', 'name', 'eanCode', 'articleCode', 'price',
                        'status', 'stockLevel', 'unlimitedStock', 'created_at', 'updated_at',
                        'concat_attributes']
    products_for_api = products[product_api_cols].copy()

    if products_snapshot is not None and len(products_snapshot) > 0:
        existing_ids = set(products_snapshot['id'].astype(str))
        new_products = products_for_api[~products_for_api['id'].isin(existing_ids)].copy()

        update_candidates = products_for_api[products_for_api['id'].isin(existing_ids)].copy()
        update_candidates = update_candidates.merge(
            products_snapshot[['id', 'optiply_id', 'concat_attributes']].rename(
                columns={'concat_attributes': 'concat_attributes_snap'}
            ), on='id', how='left'
        )
        update_products = update_candidates[
            update_candidates['concat_attributes'] != update_candidates['concat_attributes_snap']
        ].copy()
        delete_products = pd.DataFrame()
    else:
        new_products = products_for_api.copy()
        update_products = pd.DataFrame()
        delete_products = pd.DataFrame()

    print(f"Products -- DELETE: {len(delete_products)}, POST: {len(new_products)}, PATCH: {len(update_products)}")

    # DELETE
    if len(delete_products) > 0:
        for idx, row in delete_products.iterrows():
            if pd.notna(row.get('optiply_id')):
                resp = delete_optiply(api_creds, auth, int(row['optiply_id']), 'products')
                if resp.status_code in [200, 204, 404]:
                    summary['products']['deleted'] += 1
        delete_from_snapshot(delete_products, 'products', SNAPSHOT_DIR, pk='id')

    # POST
    if len(new_products) > 0:
        new_products['optiply_id'] = None
        for idx, row in new_products.iterrows():
            payload = get_product_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'products')
            if resp.status_code in [200, 201]:
                new_products.loc[idx, 'optiply_id'] = str(resp.json()['data']['id'])
                summary['products']['created'] += 1
        new_products_ok = new_products[new_products['optiply_id'].notna()]
        snapshot_records(new_products_ok, 'products', SNAPSHOT_DIR, pk='id')
        print(f"Created {summary['products']['created']} products")

    # PATCH
    if len(update_products) > 0:
        for idx, row in update_products.iterrows():
            if pd.notna(row.get('optiply_id')):
                payload = get_product_payload(row)
                resp = patch_optiply(api_creds, auth, payload, int(row['optiply_id']), 'products')
                if resp.status_code in [200, 201]:
                    summary['products']['updated'] += 1
        snapshot_records(update_products, 'products', SNAPSHOT_DIR, pk='id')
        print(f"Updated {summary['products']['updated']} products")

    gc.collect()
else:
    print("No products to process.")


## Suppliers
Derived from Products detail endpoint `productSuppliers` array.

Each product's detail response contains a `productSuppliers` list with supplier agreements.
We extract unique suppliers by `supplierAgreementNumber` and deduplicate.


In [ ]:
# Suppliers: Extract from products' product_suppliers JSON
if products is not None and len(products) > 0 and sync_suppliers:
    supplier_records = []
    seen_suppliers = set()

    for idx, row in products.iterrows():
        ps_list = parse_nested_json(row.get('product_suppliers_json', '[]'))
        source_date = row.get('source_createDate')

        for ps in ps_list:
            if not isinstance(ps, dict):
                continue
            agreement_num = ps.get('supplierAgreementNumber')
            if agreement_num is None:
                continue
            agreement_str = str(int(agreement_num)) if isinstance(agreement_num, float) else str(agreement_num)

            if agreement_str in seen_suppliers:
                continue
            seen_suppliers.add(agreement_str)

            # Delivery time in days from hours
            lead_hours = float(ps.get('manufacturingLeadTimeHour', 0) or 0)
            delivery_days = round(lead_hours / 24) if lead_hours > 0 else None

            supplier_records.append({
                'id': agreement_str,
                'remoteId': agreement_str,
                'name': str(ps.get('supplierAgreement', '')).strip()[:255] or f'Supplier {agreement_str}',
                'deliveryTime': delivery_days,
                'updated_at': datetime.datetime.utcnow().isoformat(),
            })

    if supplier_records:
        suppliers = pd.DataFrame(supplier_records)
        suppliers = suppliers.drop_duplicates(subset=['remoteId'])
        suppliers['concat_attributes'] = concat_columns(suppliers, ['name', 'deliveryTime'])
        print(f"Suppliers extracted: {len(suppliers)}")
    else:
        suppliers = None
        print("No suppliers found in product data.")
else:
    suppliers = None
    if not sync_suppliers:
        print("Suppliers sync disabled by config.")
    else:
        print("No products data for supplier extraction.")


In [ ]:
# Suppliers: Snapshot comparison and API requests
if suppliers is not None and len(suppliers) > 0:
    suppliers_snapshot = get_snapshot('suppliers', SNAPSHOT_DIR)

    if suppliers_snapshot is not None and len(suppliers_snapshot) > 0:
        existing_ids = set(suppliers_snapshot['id'].astype(str))
        new_suppliers = suppliers[~suppliers['id'].isin(existing_ids)].copy()

        update_candidates = suppliers[suppliers['id'].isin(existing_ids)].copy()
        update_candidates = update_candidates.merge(
            suppliers_snapshot[['id', 'optiply_id', 'concat_attributes']].rename(
                columns={'concat_attributes': 'concat_attributes_snap'}
            ), on='id', how='left'
        )
        update_suppliers = update_candidates[
            update_candidates['concat_attributes'] != update_candidates['concat_attributes_snap']
        ].copy()
        delete_suppliers = pd.DataFrame()
    else:
        new_suppliers = suppliers.copy()
        update_suppliers = pd.DataFrame()
        delete_suppliers = pd.DataFrame()

    print(f"Suppliers -- DELETE: {len(delete_suppliers)}, POST: {len(new_suppliers)}, PATCH: {len(update_suppliers)}")

    # POST
    if len(new_suppliers) > 0:
        new_suppliers['optiply_id'] = None
        for idx, row in new_suppliers.iterrows():
            payload = get_supplier_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'suppliers')
            if resp.status_code in [200, 201]:
                new_suppliers.loc[idx, 'optiply_id'] = str(resp.json()['data']['id'])
                summary['suppliers']['created'] += 1
        new_suppliers_ok = new_suppliers[new_suppliers['optiply_id'].notna()]
        snapshot_records(new_suppliers_ok, 'suppliers', SNAPSHOT_DIR, pk='id')
        print(f"Created {summary['suppliers']['created']} suppliers")

    # PATCH
    if len(update_suppliers) > 0:
        for idx, row in update_suppliers.iterrows():
            if pd.notna(row.get('optiply_id')):
                payload = get_supplier_payload(row)
                resp = patch_optiply(api_creds, auth, payload, int(row['optiply_id']), 'suppliers')
                if resp.status_code in [200, 201]:
                    summary['suppliers']['updated'] += 1
        snapshot_records(update_suppliers, 'suppliers', SNAPSHOT_DIR, pk='id')
        print(f"Updated {summary['suppliers']['updated']} suppliers")

    gc.collect()
else:
    print("No suppliers to process.")


## Supplier Products
Derived from Products detail endpoint `productSuppliers` array.

Each product-supplier pair becomes a SupplierProduct. The `remoteId` is a composite:
`{productNumber}_{supplierAgreementNumber}`. Price is converted to excl. VAT.


In [ ]:
# SupplierProducts: Extract from products' product_suppliers JSON
if products is not None and len(products) > 0 and sync_supplier_products:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    suppliers_snapshot = get_snapshot('suppliers', SNAPSHOT_DIR)

    sp_records = []
    for idx, row in products.iterrows():
        product_number = str(row.get('remoteId', ''))
        if not product_number or product_number == 'nan':
            continue

        # Resolve product Optiply ID
        product_optiply_id = None
        if products_snapshot is not None:
            match_p = products_snapshot[products_snapshot['id'] == product_number]
            if len(match_p) > 0 and pd.notna(match_p.iloc[0].get('optiply_id')):
                product_optiply_id = match_p.iloc[0]['optiply_id']
        if product_optiply_id is None:
            continue

        ps_list = parse_nested_json(row.get('product_suppliers_json', '[]'))
        source_date = row.get('source_createDate')

        for ps in ps_list:
            if not isinstance(ps, dict):
                continue
            agreement_num = ps.get('supplierAgreementNumber')
            if agreement_num is None:
                continue
            agreement_str = str(int(agreement_num)) if isinstance(agreement_num, float) else str(agreement_num)

            # Resolve supplier Optiply ID
            supplier_optiply_id = None
            if suppliers_snapshot is not None:
                match_s = suppliers_snapshot[suppliers_snapshot['id'] == agreement_str]
                if len(match_s) > 0 and pd.notna(match_s.iloc[0].get('optiply_id')):
                    supplier_optiply_id = match_s.iloc[0]['optiply_id']
            if supplier_optiply_id is None:
                continue

            composite_id = f"{product_number}_{agreement_str}"

            # Price excl. VAT
            raw_price = float(ps.get('price', 0) or 0)
            vat_pct = float(ps.get('vatPercent', 0) or 0)
            price_excl = round(raw_price / (1 + vat_pct / 100), 2) if vat_pct > 0 else round(raw_price, 2)

            # Delivery time
            lead_hours = float(ps.get('manufacturingLeadTimeHour', 0) or 0)
            delivery_days = round(lead_hours / 24) if lead_hours > 0 else None

            sp_records.append({
                'id': composite_id,
                'remoteId': composite_id,
                'name': str(ps.get('supplierProductName', '') or row.get('name', '')).strip()[:255],
                'skuCode': str(ps.get('supplierProductNumber', '')).strip()[:100] if ps.get('supplierProductNumber') else None,
                'price': price_excl,
                'productId': int(product_optiply_id),
                'supplierId': int(supplier_optiply_id),
                'deliveryTime': delivery_days,
                'updated_at': handle_invalid_dates(source_date) if pd.notna(source_date) else datetime.datetime.utcnow().isoformat(),
            })

    if sp_records:
        supplier_products = pd.DataFrame(sp_records)
        supplier_products = supplier_products.drop_duplicates(subset=['remoteId'])
        supplier_products['concat_attributes'] = concat_columns(
            supplier_products, ['name', 'skuCode', 'price', 'deliveryTime']
        )
        print(f"SupplierProducts extracted: {len(supplier_products)}")
    else:
        supplier_products = None
        print("No supplier products found.")
else:
    supplier_products = None
    if not sync_supplier_products:
        print("SupplierProducts sync disabled by config.")
    else:
        print("No products data for supplier product extraction.")


In [ ]:
# SupplierProducts: Snapshot comparison and API requests
if supplier_products is not None and len(supplier_products) > 0:
    sp_snapshot = get_snapshot('supplierProducts', SNAPSHOT_DIR)

    if sp_snapshot is not None and len(sp_snapshot) > 0:
        existing_ids = set(sp_snapshot['id'].astype(str))
        new_sp = supplier_products[~supplier_products['id'].isin(existing_ids)].copy()

        update_candidates = supplier_products[supplier_products['id'].isin(existing_ids)].copy()
        update_candidates = update_candidates.merge(
            sp_snapshot[['id', 'optiply_id', 'concat_attributes']].rename(
                columns={'concat_attributes': 'concat_attributes_snap'}
            ), on='id', how='left'
        )
        update_sp = update_candidates[
            update_candidates['concat_attributes'] != update_candidates['concat_attributes_snap']
        ].copy()
        delete_sp = pd.DataFrame()
    else:
        new_sp = supplier_products.copy()
        update_sp = pd.DataFrame()
        delete_sp = pd.DataFrame()

    print(f"SupplierProducts -- DELETE: {len(delete_sp)}, POST: {len(new_sp)}, PATCH: {len(update_sp)}")

    # POST
    if len(new_sp) > 0:
        new_sp['optiply_id'] = None
        for idx, row in new_sp.iterrows():
            payload = get_supplier_product_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'supplierProducts')
            if resp.status_code in [200, 201]:
                new_sp.loc[idx, 'optiply_id'] = str(resp.json()['data']['id'])
                summary['supplierProducts']['created'] += 1
        new_sp_ok = new_sp[new_sp['optiply_id'].notna()]
        snapshot_records(new_sp_ok, 'supplierProducts', SNAPSHOT_DIR, pk='id')
        print(f"Created {summary['supplierProducts']['created']} supplier products")

    # PATCH
    if len(update_sp) > 0:
        for idx, row in update_sp.iterrows():
            if pd.notna(row.get('optiply_id')):
                payload = get_supplier_product_payload(row)
                resp = patch_optiply(api_creds, auth, payload, int(row['optiply_id']), 'supplierProducts')
                if resp.status_code in [200, 201]:
                    summary['supplierProducts']['updated'] += 1
        snapshot_records(update_sp, 'supplierProducts', SNAPSHOT_DIR, pk='id')
        print(f"Updated {summary['supplierProducts']['updated']} supplier products")

    gc.collect()
else:
    print("No supplier products to process.")


## Sell Orders
Extend Commerce CustomerOrders -> Optiply SellOrders

CustomerOrders come from the `customer_orders` stream. Each order includes an `order_rows`
JSON string with line items. The ETL maps orders to SellOrders and explodes lines into
SellOrderLines. Freight rows (supplyMode == 'Freight') are excluded from lines.


In [ ]:
# SellOrders: Load from tap output
so_raw = check_sync_output('customer_orders', ['customerOrders', 'CustomerOrders', 'sellOrders'])
if so_raw is not None:
    so_raw = clean_underscore_columns(so_raw)
    print(f"Customer orders loaded: {len(so_raw)}")
else:
    print("No customer orders data found")


In [ ]:
# SellOrders: Custom mapping
COMPLETED_STATUSES = ['Delivered', 'Invoiced', 'Completed', 'FullyDelivered']

if so_raw is not None and len(so_raw) > 0 and sync_sell_orders:
    sell_orders = pd.DataFrame()
    sell_orders['id'] = so_raw['orderNumber'].astype(str)
    sell_orders['remoteId'] = so_raw['orderNumber'].astype(str)

    # Placed date
    if 'orderDate' in so_raw.columns:
        sell_orders['placed'] = so_raw['orderDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
    else:
        sell_orders['placed'] = pd.NaT

    sell_orders = sell_orders.dropna(subset=['placed'])

    # Total value (already excl. VAT per observation)
    if 'totalPrice' in so_raw.columns:
        sell_orders['totalValue'] = pd.to_numeric(
            so_raw.loc[sell_orders.index, 'totalPrice'], errors='coerce'
        ).fillna(0).round(2)
    else:
        sell_orders['totalValue'] = 0.0

    # Status handling
    if 'orderStatus' in so_raw.columns:
        status_col = so_raw.loc[sell_orders.index, 'orderStatus'].astype(str).str.strip()
    else:
        status_col = pd.Series('', index=sell_orders.index)

    # changeDate
    if 'changeDate' in so_raw.columns:
        change_dates = so_raw.loc[sell_orders.index, 'changeDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
    else:
        change_dates = pd.Series(pd.NaT, index=sell_orders.index)

    sell_orders['updated_at'] = change_dates

    # Completed: set changeDate if status is a completed status
    sell_orders['completed'] = None
    completed_mask = status_col.isin(COMPLETED_STATUSES)
    sell_orders.loc[completed_mask, 'completed'] = change_dates[completed_mask]

    # Deleted: set changeDate if status is Cancelled
    sell_orders['deleted_at'] = None
    cancelled_mask = status_col == 'Cancelled'
    sell_orders.loc[cancelled_mask, 'deleted_at'] = change_dates[cancelled_mask]

    # Keep order_rows JSON for SellOrderLines extraction
    if 'order_rows' in so_raw.columns:
        sell_orders['order_rows_json'] = so_raw.loc[sell_orders.index, 'order_rows'].values
    else:
        sell_orders['order_rows_json'] = '[]'

    sell_orders = sell_orders.drop_duplicates(subset=['remoteId'])
    sell_orders['concat_attributes'] = concat_columns(
        sell_orders, ['totalValue', 'completed', 'deleted_at']
    )

    print(f"Sell orders mapped: {len(sell_orders)}")
else:
    sell_orders = None
    if not sync_sell_orders:
        print("Sell orders sync disabled by config.")
    else:
        print("No customer orders data found.")


In [ ]:
# SellOrderLines: Extract from sell_orders' order_rows JSON
if sell_orders is not None and len(sell_orders) > 0 and sync_sell_orders:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)

    sol_records = []
    for idx, row in sell_orders.iterrows():
        order_number = str(row.get('remoteId', ''))
        if not order_number or order_number == 'nan':
            continue

        order_rows = parse_nested_json(row.get('order_rows_json', '[]'))

        for line in order_rows:
            if not isinstance(line, dict):
                continue

            # Filter out freight rows
            supply_mode = line.get('supplyMode', '')
            if supply_mode == 'Freight':
                continue

            product_info = line.get('product', {}) or {}
            product_number = product_info.get('productNumber')
            if not product_number:
                continue

            position = line.get('position', 0)
            line_remote_id = f"{order_number}-{position}"

            sales_data = line.get('salesData', {}) or {}
            qty = int(float(sales_data.get('quantity', 0) or 0))
            unit_price = float(sales_data.get('unitPrice', 0) or 0)
            subtotal = round(qty * unit_price, 2)

            # Resolve product Optiply ID
            product_optiply_id = None
            if products_snapshot is not None:
                match_p = products_snapshot[products_snapshot['id'] == str(product_number)]
                if len(match_p) > 0 and pd.notna(match_p.iloc[0].get('optiply_id')):
                    product_optiply_id = match_p.iloc[0]['optiply_id']
            if product_optiply_id is None:
                continue

            # changeDate from line or order
            line_change = line.get('changeDate')
            updated_at = None
            if line_change and pd.notna(line_change):
                dt = handle_invalid_dates(line_change)
                updated_at = None if pd.isna(dt) else str(dt)
            if updated_at is None:
                updated_at = str(row.get('updated_at', '')) if pd.notna(row.get('updated_at')) else None

            sol_records.append({
                'id': line_remote_id,
                'remoteId': line_remote_id,
                'quantity': qty,
                'productId': int(product_optiply_id),
                'sellOrderId': order_number,  # Will be resolved to optiply_id below
                'subtotalValue': subtotal,
                'updated_at': updated_at,
            })

    if sol_records:
        sell_order_lines = pd.DataFrame(sol_records)
        sell_order_lines = sell_order_lines.drop_duplicates(subset=['remoteId'])
        print(f"Sell order lines extracted: {len(sell_order_lines)}")
    else:
        sell_order_lines = None
        print("No sell order lines extracted.")
else:
    sell_order_lines = None
    print("No sell orders data for line extraction.")


In [ ]:
# SellOrders: Nest lines + Snapshot comparison + API requests
# Optiply SellOrders use get_sell_order_withlines_payload which embeds orderLines
if sell_orders is not None and len(sell_orders) > 0:
    so_snapshot = get_snapshot('sellOrders', SNAPSHOT_DIR)

    # Prepare sell orders for API (exclude helper columns)
    so_api_cols = ['id', 'remoteId', 'placed', 'totalValue', 'completed', 'deleted_at', 'updated_at', 'concat_attributes']
    so_for_api = sell_orders[so_api_cols].copy()

    # Attach order lines as nested list for get_sell_order_withlines_payload
    if sell_order_lines is not None and len(sell_order_lines) > 0:
        # Group lines by sellOrderId (still remote order number at this point)
        lines_grouped = sell_order_lines.groupby('sellOrderId').apply(
            lambda x: x[['quantity', 'subtotalValue', 'productId']].to_dict('records')
        )
        so_for_api = so_for_api.merge(
            lines_grouped.rename('orderLines'),
            left_on='remoteId', right_index=True, how='left'
        )
    else:
        so_for_api['orderLines'] = so_for_api['remoteId'].apply(lambda x: [])

    # Fill NaN orderLines with empty list
    so_for_api['orderLines'] = so_for_api['orderLines'].apply(
        lambda x: x if isinstance(x, list) else []
    )

    if so_snapshot is not None and len(so_snapshot) > 0:
        existing_ids = set(so_snapshot['id'].astype(str))
        new_sell_orders = so_for_api[~so_for_api['id'].isin(existing_ids)].copy()

        update_candidates = so_for_api[so_for_api['id'].isin(existing_ids)].copy()
        update_candidates = update_candidates.merge(
            so_snapshot[['id', 'optiply_id', 'concat_attributes']].rename(
                columns={'concat_attributes': 'concat_attributes_snap'}
            ), on='id', how='left'
        )
        update_sell_orders = update_candidates[
            update_candidates['concat_attributes'] != update_candidates['concat_attributes_snap']
        ].copy()
        delete_sell_orders = pd.DataFrame()
    else:
        new_sell_orders = so_for_api.copy()
        update_sell_orders = pd.DataFrame()
        delete_sell_orders = pd.DataFrame()

    print(f"SellOrders -- DELETE: {len(delete_sell_orders)}, POST: {len(new_sell_orders)}, PATCH: {len(update_sell_orders)}")

    # POST (uses get_sell_order_withlines_payload which embeds orderLines)
    if len(new_sell_orders) > 0:
        new_sell_orders['optiply_id'] = None
        for idx, row in new_sell_orders.iterrows():
            payload = get_sell_order_withlines_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'sellOrders')
            if resp.status_code in [200, 201]:
                new_sell_orders.loc[idx, 'optiply_id'] = str(resp.json()['data']['id'])
                summary['sellOrders']['created'] += 1
        new_so_ok = new_sell_orders[new_sell_orders['optiply_id'].notna()]
        snapshot_records(new_so_ok, 'sellOrders', SNAPSHOT_DIR, pk='id')
        print(f"Created {summary['sellOrders']['created']} sell orders")

    # PATCH
    if len(update_sell_orders) > 0:
        for idx, row in update_sell_orders.iterrows():
            if pd.notna(row.get('optiply_id')):
                payload = get_sell_order_withlines_payload(row)
                resp = patch_optiply(api_creds, auth, payload, int(row['optiply_id']), 'sellOrders')
                if resp.status_code in [200, 201]:
                    summary['sellOrders']['updated'] += 1
        snapshot_records(update_sell_orders, 'sellOrders', SNAPSHOT_DIR, pk='id')
        print(f"Updated {summary['sellOrders']['updated']} sell orders")

    gc.collect()
else:
    print("No sell orders to process.")


## Buy Orders
Extend Commerce PurchaseOrders -> Optiply BuyOrders

**Bidirectional guard:** Orders where `externalOrderNumber` contains 'optiply' (case-insensitive) are filtered out to prevent feedback loops when the target writes back to Extend.

In [ ]:
# Load purchase orders from Extend tap output
po_raw = check_sync_output('purchase_orders', ['purchaseOrders', 'PurchaseOrders', 'buyOrders'])
if po_raw is not None:
    po_raw = clean_underscore_columns(po_raw)
    print(f"Purchase orders loaded: {len(po_raw)}")
else:
    print("No purchase orders data found")

# Map Extend PurchaseOrders to Optiply BuyOrders
if po_raw is not None and len(po_raw) > 0 and sync_buy_orders:
    buy_orders = pd.DataFrame()
    buy_orders['id'] = po_raw['purchaseNumber'].astype(str)
    buy_orders['remoteId'] = po_raw['purchaseNumber'].astype(str)

    # Placed date from createDate
    if 'createDate' in po_raw.columns:
        buy_orders['placed'] = po_raw['createDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
    else:
        buy_orders['placed'] = pd.NaT

    buy_orders = buy_orders.dropna(subset=['placed'])

    # Supplier remote ID
    if 'supplierNumber' in po_raw.columns:
        buy_orders['supplierRemoteId'] = po_raw.loc[buy_orders.index, 'supplierNumber'].astype(str)
    else:
        buy_orders['supplierRemoteId'] = None

    # External order number (for bidirectional guard)
    if 'externalOrderNumber' in po_raw.columns:
        buy_orders['externalOrderNumber'] = po_raw.loc[buy_orders.index, 'externalOrderNumber'].astype(str)
    else:
        buy_orders['externalOrderNumber'] = ''

    # Reference (for bidirectional guard)
    if 'reference' in po_raw.columns:
        buy_orders['reference'] = po_raw.loc[buy_orders.index, 'reference'].astype(str)
    else:
        buy_orders['reference'] = ''

    # Status fields
    if 'status' in po_raw.columns:
        buy_orders['status'] = po_raw.loc[buy_orders.index, 'status'].astype(str)
    else:
        buy_orders['status'] = ''

    if 'isOpen' in po_raw.columns:
        buy_orders['isOpen'] = po_raw.loc[buy_orders.index, 'isOpen']
    else:
        buy_orders['isOpen'] = True

    # Keep rows JSON for totalValue and completed computation
    if 'rows' in po_raw.columns:
        buy_orders['rows_json'] = po_raw.loc[buy_orders.index, 'rows']
    else:
        buy_orders['rows_json'] = '[]'

    # Total value: sum(qty * unitPrice) across all rows, excl. VAT
    def compute_bo_total(row_idx):
        row = po_raw.loc[row_idx]
        rows_val = row.get('rows', '[]')
        parsed_rows = parse_nested_json(rows_val)
        total = 0.0
        for r in parsed_rows:
            if not isinstance(r, dict):
                continue
            product_unit = r.get('purchaseDataProductUnit', {}) or {}
            qty = float(product_unit.get('quantity', 0) or 0)
            unit_price = float(product_unit.get('unitPrice', 0) or 0)
            total += qty * unit_price
        return round(total, 2)

    buy_orders['totalValue'] = buy_orders.index.map(compute_bo_total)

    # Completed: Received AND !isOpen -> latest statusChangeDate from rows
    buy_orders['completed'] = buy_orders.apply(
        lambda row: get_extend_status_completed(
            row['status'], row['isOpen'], False, row['rows_json']
        ), axis=1
    )

    # Deleted at: Cancelled status
    buy_orders['deleted_at'] = buy_orders['status'].apply(get_deleted_at)

    # updated_at: use createDate (no explicit updated_at in Extend)
    buy_orders['updated_at'] = buy_orders['placed']

    print(f"Buy orders mapped (pre-filter): {len(buy_orders)}")
else:
    buy_orders = None
    if not sync_buy_orders:
        print("Buy orders sync disabled by config.")
    else:
        print("No purchase orders data found.")

In [ ]:
# BuyOrders: Bidirectional guard -- filter out orders created by Optiply
if buy_orders is not None and len(buy_orders) > 0:
    pre_filter_count = len(buy_orders)

    # Filter out orders where externalOrderNumber or reference contains 'optiply'
    optiply_mask_ext = buy_orders['externalOrderNumber'].str.lower().str.contains('optiply', na=False)
    optiply_mask_ref = buy_orders['reference'].str.lower().str.contains('optiply', na=False)
    optiply_mask = optiply_mask_ext | optiply_mask_ref
    buy_orders = buy_orders[~optiply_mask]

    filtered_count = pre_filter_count - len(buy_orders)
    if filtered_count > 0:
        print(f"Bidirectional guard: Filtered out {filtered_count} Optiply-created buy orders")

    # Filter out Cancelled orders (will be handled as deletes)
    cancelled_mask = buy_orders['deleted_at'].notna()
    cancelled_orders = buy_orders[cancelled_mask]
    buy_orders_active = buy_orders[~cancelled_mask]
    print(f"Cancelled orders: {len(cancelled_orders)}, Active orders: {len(buy_orders_active)}")

    # Resolve supplier Optiply ID
    suppliers_snapshot = get_snapshot('suppliers', SNAPSHOT_DIR)
    if suppliers_snapshot is not None and 'supplierRemoteId' in buy_orders_active.columns:
        buy_orders_active = buy_orders_active.merge(
            suppliers_snapshot[['id', 'optiply_id']].rename(
                columns={'id': 'supplierRemoteId', 'optiply_id': 'supplierId'}
            ),
            on='supplierRemoteId', how='left'
        )
    else:
        buy_orders_active['supplierId'] = None

    buy_orders_active = buy_orders_active[buy_orders_active['supplierId'].notna()]
    print(f"Buy orders after FK resolution: {len(buy_orders_active)}")
    
    # Reassign for downstream use
    buy_orders = buy_orders_active

In [ ]:
# BuyOrders: Snapshot comparison
if buy_orders is not None and len(buy_orders) > 0:
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
    bo_diff = snapshot_records(
        buy_orders[['id', 'remoteId', 'totalValue', 'supplierId']],
        'buyOrders', SNAPSHOT_DIR, pk='id'
    )

    if bo_snapshot is not None:
        existing_ids = set(bo_snapshot['id'].astype(str))
        current_bo_ids = set(buy_orders['id'].astype(str))
        removed_bo_ids = existing_ids - current_bo_ids

        # Also mark cancelled orders for deletion
        if cancelled_orders is not None and len(cancelled_orders) > 0:
            cancelled_ids = set(cancelled_orders['id'].astype(str))
            removed_bo_ids = removed_bo_ids | (cancelled_ids & existing_ids)

        delete_buy_orders = bo_snapshot[bo_snapshot['id'].isin(removed_bo_ids)] if removed_bo_ids else pd.DataFrame()
        new_buy_orders = buy_orders[~buy_orders['id'].isin(existing_ids)]
        update_buy_orders = buy_orders[buy_orders['id'].isin(existing_ids)]
    else:
        delete_buy_orders = pd.DataFrame()
        new_buy_orders = buy_orders
        update_buy_orders = pd.DataFrame()

    print(f"BuyOrders -- DELETE: {len(delete_buy_orders)}, POST: {len(new_buy_orders)}, PATCH: {len(update_buy_orders)}")
else:
    delete_buy_orders = pd.DataFrame()
    new_buy_orders = pd.DataFrame()
    update_buy_orders = pd.DataFrame()

In [ ]:
# BuyOrders: Handle supplierId changes (delete + recreate if supplier changed)
if len(update_buy_orders) > 0:
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
    if bo_snapshot is not None and 'supplierId' in bo_snapshot.columns:
        supplier_changed = []
        for idx, row in update_buy_orders.iterrows():
            match = bo_snapshot[bo_snapshot['id'] == row['id']]
            if len(match) > 0:
                old_supplier = str(match.iloc[0].get('supplierId', ''))
                new_supplier = str(row.get('supplierId', ''))
                if old_supplier != new_supplier and old_supplier != 'nan' and new_supplier != 'nan':
                    supplier_changed.append(row['id'])

        if supplier_changed:
            print(f"Supplier changed on {len(supplier_changed)} buy orders -- will delete + recreate")
            changed_mask = update_buy_orders['id'].isin(supplier_changed)
            changed_rows = update_buy_orders[changed_mask]

            changed_in_snapshot = bo_snapshot[bo_snapshot['id'].isin(supplier_changed)]
            delete_buy_orders = pd.concat([delete_buy_orders, changed_in_snapshot], ignore_index=True)
            new_buy_orders = pd.concat([new_buy_orders, changed_rows], ignore_index=True)
            update_buy_orders = update_buy_orders[~changed_mask]

            print(f"After supplier change handling -- DELETE: {len(delete_buy_orders)}, POST: {len(new_buy_orders)}, PATCH: {len(update_buy_orders)}")

In [ ]:
# BuyOrders: Build payloads
new_bo_payloads = []
for idx, row in new_buy_orders.iterrows():
    new_bo_payloads.append(get_buy_order_payload(row))

update_bo_payloads = []
for idx, row in update_buy_orders.iterrows():
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
    if bo_snapshot is not None:
        match = bo_snapshot[bo_snapshot['id'] == row['id']]
        if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
            update_bo_payloads.append({
                'optiply_id': match.iloc[0]['optiply_id'],
                'payload': get_buy_order_payload(row),
            })

print(f"BuyOrder payloads -- POST: {len(new_bo_payloads)}, PATCH: {len(update_bo_payloads)}")

In [ ]:
# BuyOrders: Execute DELETE > POST > PATCH
try:
    # DELETE
    if len(delete_buy_orders) > 0:
        for idx, row in delete_buy_orders.iterrows():
            if pd.notna(row.get('optiply_id')):
                resp = delete_optiply(api_creds, auth, row['optiply_id'], 'buyOrders')
                if resp.status_code in [200, 204, 404]:
                    summary['buyOrders']['deleted'] += 1
        delete_from_snapshot(delete_buy_orders, 'buyOrders', SNAPSHOT_DIR, pk='id')
        print(f"Deleted {summary['buyOrders']['deleted']} buy orders")

    # POST
    if len(new_buy_orders) > 0:
        bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
        for idx, row in new_buy_orders.iterrows():
            payload = get_buy_order_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'buyOrders')
            if resp.status_code in [200, 201]:
                optiply_id = str(resp.json()['data']['id'])
                if bo_snapshot is not None:
                    bo_snapshot.loc[bo_snapshot['id'] == row['id'], 'optiply_id'] = optiply_id
                summary['buyOrders']['created'] += 1
        if bo_snapshot is not None:
            bo_snapshot.to_csv(f"{SNAPSHOT_DIR}/buyOrders.snapshot.csv", index=False)
        print(f"Created {summary['buyOrders']['created']} buy orders")

    # PATCH
    if len(update_buy_orders) > 0:
        bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
        for idx, row in update_buy_orders.iterrows():
            if bo_snapshot is not None:
                match = bo_snapshot[bo_snapshot['id'] == row['id']]
                if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
                    optiply_id = match.iloc[0]['optiply_id']
                    payload = get_buy_order_payload(row)
                    resp = patch_optiply(api_creds, auth, payload, optiply_id, 'buyOrders')
                    if resp.status_code in [200, 201]:
                        summary['buyOrders']['updated'] += 1
        print(f"Updated {summary['buyOrders']['updated']} buy orders")
finally:
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
    if bo_snapshot is not None:
        bo_snapshot.to_csv(f"{SNAPSHOT_DIR}/buyOrders.snapshot.csv", index=False)

gc.collect()

### Buy Order Lines
Extend PurchaseOrder rows -> Optiply BuyOrderLines

Lines are extracted from the `rows` JSON array embedded in each PurchaseOrder record.
The `remoteId` is constructed as `{purchaseNumber}-{position}` (e.g. `RP-10748-10`).

In [ ]:
# BuyOrderLines: Extract from rows nested in purchase orders
if po_raw is not None and len(po_raw) > 0 and sync_buy_orders:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)

    buy_order_lines_data = []
    for idx, row in po_raw.iterrows():
        bo_remote_id = str(row.get('purchaseNumber', ''))
        if not bo_remote_id or bo_remote_id == 'nan':
            continue

        # Bidirectional guard: skip Optiply-created orders
        ext_order = str(row.get('externalOrderNumber', ''))
        reference = str(row.get('reference', ''))
        if 'optiply' in ext_order.lower() or 'optiply' in reference.lower():
            continue

        # Skip cancelled orders
        status = str(row.get('status', '')).strip()
        if status == 'Cancelled':
            continue

        # Resolve buy order Optiply ID
        bo_optiply_id = None
        if bo_snapshot is not None:
            match = bo_snapshot[bo_snapshot['id'] == bo_remote_id]
            if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
                bo_optiply_id = match.iloc[0]['optiply_id']
        if bo_optiply_id is None:
            continue

        # Parse rows JSON
        rows_val = row.get('rows', '[]')
        parsed_rows = parse_nested_json(rows_val)

        for po_row in parsed_rows:
            if not isinstance(po_row, dict):
                continue

            product_number = str(po_row.get('productNumber', ''))
            if not product_number or product_number == 'nan':
                continue

            position = po_row.get('position', 0)

            # Resolve productId
            optiply_pid = None
            if products_snapshot is not None:
                match_p = products_snapshot[products_snapshot['id'] == product_number]
                if len(match_p) > 0 and pd.notna(match_p.iloc[0].get('optiply_id')):
                    optiply_pid = match_p.iloc[0]['optiply_id']
            if optiply_pid is None:
                continue

            line_remote_id = f"{bo_remote_id}-{position}"

            # Quantity from purchaseDataPurchaseUnit
            purchase_unit = po_row.get('purchaseDataPurchaseUnit', {}) or {}
            qty = int(float(purchase_unit.get('quantity', 0) or 0))

            # Subtotal from purchaseDataProductUnit (qty * unitPrice)
            product_unit = po_row.get('purchaseDataProductUnit', {}) or {}
            pu_qty = float(product_unit.get('quantity', 0) or 0)
            unit_price = float(product_unit.get('unitPrice', 0) or 0)
            subtotal = round(pu_qty * unit_price, 2)

            # Updated at: expectedDeliveryDate or statusChangeDate fallback
            updated_at = None
            expected_delivery = po_row.get('expectedDeliveryDate')
            if expected_delivery and pd.notna(expected_delivery):
                dt = handle_invalid_dates(expected_delivery)
                updated_at = None if pd.isna(dt) else str(dt)
            if updated_at is None:
                scd = po_row.get('statusChangeDate')
                if scd and pd.notna(scd):
                    dt = handle_invalid_dates(scd)
                    updated_at = None if pd.isna(dt) else str(dt)

            buy_order_lines_data.append({
                'id': line_remote_id,
                'remoteId': line_remote_id,
                'buyOrderId': int(bo_optiply_id),
                'productId': int(optiply_pid),
                'quantity': qty,
                'subtotalValue': subtotal,
                'updated_at': updated_at,
            })

    if buy_order_lines_data:
        bol_df = pd.DataFrame(buy_order_lines_data)
        print(f"Buy order lines extracted: {len(bol_df)}")
    else:
        bol_df = None
        print("No buy order lines extracted from purchase orders.")
else:
    bol_df = None
    print("No purchase orders data for buy order lines.")

In [ ]:
# BuyOrderLines: FK references already resolved during extraction
# buyOrderId and productId were resolved from bo_snapshot and products_snapshot
if bol_df is not None:
    print(f"Buy order lines with resolved FKs: {len(bol_df)}")
    print(f"  Unique buyOrderIds: {bol_df['buyOrderId'].nunique()}")
    print(f"  Unique productIds: {bol_df['productId'].nunique()}")

In [ ]:
# BuyOrderLines: Snapshot comparison
if bol_df is not None and len(bol_df) > 0:
    bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)
    bol_diff = snapshot_records(bol_df, 'buyOrderLines', SNAPSHOT_DIR, pk='id')

    if bol_snapshot is not None:
        existing_ids = set(bol_snapshot['id'].astype(str))
        current_bol_ids = set(bol_df['id'].astype(str))
        removed_bol_ids = existing_ids - current_bol_ids

        delete_bol = bol_snapshot[bol_snapshot['id'].isin(removed_bol_ids)] if removed_bol_ids else pd.DataFrame()
        new_bol = bol_diff[~bol_diff['id'].isin(existing_ids)] if bol_diff is not None and len(bol_diff) > 0 else pd.DataFrame()
        update_bol = bol_diff[bol_diff['id'].isin(existing_ids)] if bol_diff is not None and len(bol_diff) > 0 else pd.DataFrame()
    else:
        delete_bol = pd.DataFrame()
        new_bol = bol_df
        update_bol = pd.DataFrame()

    print(f"BuyOrderLines -- DELETE: {len(delete_bol)}, POST: {len(new_bol)}, PATCH: {len(update_bol)}")
else:
    delete_bol = pd.DataFrame()
    new_bol = pd.DataFrame()
    update_bol = pd.DataFrame()

In [ ]:
# BuyOrderLines: Build payloads
new_bol_payloads = []
for idx, row in new_bol.iterrows():
    new_bol_payloads.append(get_buy_order_line_payload(row))

update_bol_payloads = []
for idx, row in update_bol.iterrows():
    bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)
    if bol_snapshot is not None:
        match = bol_snapshot[bol_snapshot['id'] == row['id']]
        if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
            update_bol_payloads.append({
                'optiply_id': match.iloc[0]['optiply_id'],
                'payload': get_buy_order_line_payload(row),
            })

print(f"BuyOrderLine payloads -- POST: {len(new_bol_payloads)}, PATCH: {len(update_bol_payloads)}")

In [ ]:
# BuyOrderLines: Execute DELETE > POST > PATCH
try:
    # DELETE
    if len(delete_bol) > 0:
        for idx, row in delete_bol.iterrows():
            if pd.notna(row.get('optiply_id')):
                resp = delete_optiply(api_creds, auth, row['optiply_id'], 'buyOrderLines')
                if resp.status_code in [200, 204, 404]:
                    summary['buyOrderLines']['deleted'] += 1
        delete_from_snapshot(delete_bol, 'buyOrderLines', SNAPSHOT_DIR, pk='id')
        print(f"Deleted {summary['buyOrderLines']['deleted']} buy order lines")

    # POST
    if len(new_bol) > 0:
        bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)
        for idx, row in new_bol.iterrows():
            payload = get_buy_order_line_payload(row)
            resp = post_optiply(api_creds, auth, payload, 'buyOrderLines')
            if resp.status_code in [200, 201]:
                optiply_id = str(resp.json()['data']['id'])
                if bol_snapshot is not None:
                    bol_snapshot.loc[bol_snapshot['id'] == row['id'], 'optiply_id'] = optiply_id
                summary['buyOrderLines']['created'] += 1
        if bol_snapshot is not None:
            bol_snapshot.to_csv(f"{SNAPSHOT_DIR}/buyOrderLines.snapshot.csv", index=False)
        print(f"Created {summary['buyOrderLines']['created']} buy order lines")

    # PATCH
    if len(update_bol) > 0:
        bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)
        for idx, row in update_bol.iterrows():
            if bol_snapshot is not None:
                match = bol_snapshot[bol_snapshot['id'] == row['id']]
                if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
                    optiply_id = match.iloc[0]['optiply_id']
                    payload = get_buy_order_line_payload(row)
                    resp = patch_optiply(api_creds, auth, payload, optiply_id, 'buyOrderLines')
                    if resp.status_code in [200, 201]:
                        summary['buyOrderLines']['updated'] += 1
        print(f"Updated {summary['buyOrderLines']['updated']} buy order lines")
finally:
    bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)
    if bol_snapshot is not None:
        bol_snapshot.to_csv(f"{SNAPSHOT_DIR}/buyOrderLines.snapshot.csv", index=False)

gc.collect()

In [ ]:
print("=" * 50)
print("ETL Summary -- Extend Commerce")
print("=" * 50)
for entity, counts in summary.items():
    total = sum(counts.values())
    if total > 0:
        print(f"  {entity}: {counts}")
    else:
        print(f"  {entity}: no changes")
print("=" * 50)
